In [ ]:
from pyspark.sql.functions import *

In [ ]:
raw_fire_df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/databricks-datasets/learning-spark-v2/sf-fire/sf-fire-calls.csv")

In [ ]:
display(raw_fire_df)

In [ ]:
renamed_fire_df = raw_fire_df \
    .withColumnRenamed("Call Number", "CallNumber") \
    .withColumnRenamed("Unit ID", "UnitID") \
    .withColumnRenamed("Incident Number", "IncidentNumber") \
    .withColumnRenamed("Call Date", "CallDate") \
    .withColumnRenamed("Watch Date", "WatchDate") \
    .withColumnRenamed("Call Final Disposition", "CallFinalDisposition") \
    .withColumnRenamed("Available DtTm", "AvailableDtTm") \
    .withColumnRenamed("Zipcode of Incident", "Zipcode") \
    .withColumnRenamed("Station Area", "StationArea") \
    .withColumnRenamed("Final Priority", "FinalPriority") \
    .withColumnRenamed("ALS Unit", "ALS Unit") \
    .withColumnRenamed("Call Type Group", "CallTypeGroup") \
    .withColumnRenamed("Unit sequence in call dispatch", "UnitSequenceInCallDispatch") \
    .withColumnRenamed("Fire Prevention District", "FirePreventionDistrict") \
    .withColumnRenamed("Supervisor District", "SupervisorDistrict")         

In [ ]:
display(renamed_fire_df)

In [ ]:
fire_df = renamed_fire_df \
    .withColumn("CallDate", to_date("CallDate", "MM/dd/yyyy")) \
    .withColumn("WatchDate", to_date("WatchDate", "MM/dd/yyyy")) \
    .withColumn("AvailableDtTm", to_timestamp("AvailableDtTm", "MM/dd/yyyy hh:mm:ss a")) \
    .withColumn("Delay", round("Delay", 2))
    

In [ ]:
display(fire_df)

In [ ]:
fire_df.printSchema()

In [ ]:
fire_df.cache()

Q1. How many distinct types of calls were made to the Fire Department?

In [ ]:
q1_df = fire_df.where("CallType is not null") \
            .select("CallType") \
            .distinct()

display(q1_df.count())

%md
#### Q2. What were distinct types of calls made to the Fire Department?

In [ ]:
q2_df = fire_df.where("CallType is not null") \
            .select(expr("CallType as distinct_call_type")) \
            .distinct()

q2_df.show()

Q3. Find out all response for delayed times greater than 5 mins?

In [ ]:
fire_df.where("Delay > 5") \
    .select("CallNumber", "Delay") \
    .show()


Q4. What were the most common call types?

In [ ]:
q4_df = fire_df.select("CallType") \
    .where("CallType is not null") \
    .groupBy("CallType") \
    .count() \
    .orderBy("count", ascending=False)

display(q4_df)

%md
Q5. What zip codes accounted for most common calls?

In [ ]:
q5_df = fire_df.select("CallType", "ZipCode") \
    .where("CallType is not null") \
    .groupBy("CallType", "ZipCode") \
    .count() \
    .orderBy("count", ascending=False)

q5_df.show()

%md
Q6. What San Francisco neighborhoods are in the zip codes 94102 and 94103?

In [ ]:
q6_df = fire_df.select("Neighborhood") \
    .distinct() \
    .where("ZipCode in (94102, 94103)")

q6_df.show()

Q7. What was the sum of all call alarms, average, min, and max of the call response time?

In [ ]:
q7_df = fire_df.select(
    sum("NumAlarms"),
    avg("Delay"),
    min("Delay"),
    max("Delay")
)

q7_df.show()

Q8. How Many distinct years of data is in the data set?

In [ ]:
q8_df = fire_df.select(
    year("CallDate").alias("year")) \
    .distinct() \
    .orderBy("year")
display(q8_df)

Q9. What week of the year in 2018 had the most fire calls?

In [ ]:
q9_df = fire_df.select("CallDate") \
            .where("year(CallDate) = 2018") \
            .groupBy(weekofyear("CallDate")) \
            .count() \
            .orderBy("count", ascending=False)

display(q9_df)

Q10. What neighborhoods in San Francisco had the worst response time in 2018?

In [ ]:
q10_df = fire_df.select("Neighborhood", "Delay") \
    .where("year(CallDate) = 2018") \
    .orderBy("Delay", ascending=False)

q10_df.show()